In [1]:
from google.colab import files
uploaded = files.upload()

import os
pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("PDFs uploaded:", pdf_paths)
if not pdf_paths:
    raise ValueError("Please upload the two PDF files.")


Saving clgdataset.pdf to clgdataset.pdf
Saving kaggle.pdf to kaggle.pdf
PDFs uploaded: ['clgdataset.pdf', 'kaggle.pdf']


In [9]:
!pip -q install --upgrade pymupdf

import pymupdf
import re
import os
import math
import statistics
from collections import Counter

def extract_pages(pdf_path):
    doc = pymupdf.open(pdf_path)
    pages = []

    for i, page in enumerate(doc, start=1):
        text = page.get_text("text")

        # Normalize line endings and repeated whitespace
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        pages.append({
            "page": i,
            "text": text
        })

    doc.close()
    return pages


documents = {
    os.path.basename(p): extract_pages(p)
    for p in pdf_paths
}

for name, pages in documents.items():
    chars = sum(len(p["text"]) for p in pages)
    print(
        f"{name}: {len(pages)} pages, "
        f"{chars:,} extracted characters"
    )

clgdataset.pdf: 27 pages, 14,060 extracted characters
kaggle.pdf: 5 pages, 8,006 extracted characters


In [22]:
import pandas as pd
from IPython.display import display

TARGET_CHARS = 900
OVERLAP_CHARS = 120
STEP_SIZE = TARGET_CHARS - OVERLAP_CHARS

def fixed_size_chunking(pages, size=TARGET_CHARS, overlap=OVERLAP_CHARS):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Chunk size must be positive and overlap must be smaller than chunk size.")

    full_text = "\n\n".join(
        page["text"].strip()
        for page in pages
        if page["text"].strip()
    )

    chunks = []
    start = 0

    while start < len(full_text):
        end = min(start + size, len(full_text))
        chunk_text = full_text[start:end]

        chunks.append({
            "text": chunk_text,
            "char_count": len(chunk_text),
            "start_char": start,
            "end_char": end
        })

        if end == len(full_text):
            break

        start += size - overlap

    return chunks


fixed_size_results = []

for filename, pages in documents.items():
    chunks = fixed_size_chunking(pages)

    print("\n" + "=" * 100)
    print(f"DOCUMENT: {filename}")
    print(f"TOTAL CHUNKS: {len(chunks)}")
    print("=" * 100)

    for chunk_id, chunk in enumerate(chunks, start=1):
        record = {
            "document": filename,
            "chunk_id": chunk_id,
            "method": "fixed-size",
            "start_char": chunk["start_char"],
            "end_char": chunk["end_char"],
            "text": chunk["text"],
            "char_count": chunk["char_count"]
        }

        fixed_size_results.append(record)

        print(f"\n--- CHUNK {chunk_id} ---")
        print(f"Characters: {chunk['char_count']}")
        print(f"Character positions: {chunk['start_char']}–{chunk['end_char'] - 1}")
        print("-" * 80)
        print(chunk["text"])
        print("-" * 80)

fixed_size_df = pd.DataFrame(fixed_size_results)

fixed_size_df.to_csv("fixed_size_chunks.csv", index=False)

print("\n" + "=" * 100)
print("CSV SAVED: fixed_size_chunks.csv")
print(f"TOTAL CHUNKS ACROSS BOTH DOCUMENTS: {len(fixed_size_df)}")
print("=" * 100)

print("\nFIRST 5 ROWS OF THE CSV")
display(fixed_size_df.head(5))


DOCUMENT: clgdataset.pdf
TOTAL CHUNKS: 18

--- CHUNK 1 ---
Characters: 900
Character positions: 0–899
--------------------------------------------------------------------------------
Definition and philosophy
Rational Agents
Nature of Environment
Agents and Environments
Roshni Chakraborty
Dept. Of Information Technology
ABV-IIITM Gwalior
September 1, 2026
1 / 27

Definition and philosophy
Rational Agents
Nature of Environment
Outline I
1
Definition and philosophy
Quantify Intelligence
Thinking Humanly
2
Rational Agents
Ideal Rational Agents
Vaccum Cleaner Agent
3
Nature of Environment
Goal based Agents
2 / 27

Definition and philosophy
Rational Agents
Nature of Environment
Quantify Intelligence
Thinking Humanly
Artificial Intelligence
Definition and philosophy
How to quantify intelligence ?
Are humans intelligent?
replicating human behavior is an early hallmark of intelligence
Are humans always intelligent?
Can non-human behavior be intelligent?
Definition
The study of how to make com

,document,chunk_id,method,start_char,end_char,text,char_count
0,clgdataset.pdf,1,fixed-size,0,900,Definition and philosophy\nRational Agents\nNa...,900
1,clgdataset.pdf,2,fixed-size,780,1680,inition\nThe study of how to make computers do...,900
2,clgdataset.pdf,3,fixed-size,1560,2460,\nRational Agents\nNature of Environment\nQuan...,900
3,clgdataset.pdf,4,fixed-size,2340,3240,ight thing\nbased on what it can perceive\nand...,900
4,clgdataset.pdf,5,fixed-size,3120,4020,erformance\nmeasure\non the basis of evidence ...,900


In [44]:
fixed_size_df = pd.DataFrame(fixed_size_results)

kaggle_fixed_size_df = fixed_size_df[
    fixed_size_df["document"] == "kaggle.pdf"
].copy()

kaggle_fixed_size_df.to_csv(
    "kaggle_fixed_size_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("CSV SAVED: kaggle_fixed_size_chunks.csv")
print(f"TOTAL KAGGLE CHUNKS: {len(kaggle_fixed_size_df)}")
display(kaggle_fixed_size_df.head(5))

CSV SAVED: kaggle_fixed_size_chunks.csv
TOTAL KAGGLE CHUNKS: 11


,document,chunk_id,method,start_char,end_char,text,char_count
18,kaggle.pdf,1,fixed-size,0,900,Peer Review Framework for Predictive Analytics...,900
19,kaggle.pdf,2,fixed-size,780,1680,"\nbefore a typhoon hit, based on the predicted...",900
20,kaggle.pdf,3,fixed-size,1560,2460,l will trigger or inform as part of the 2021-2...,900
21,kaggle.pdf,4,fixed-size,2340,3240,"in the ethical review, more information about...",900
22,kaggle.pdf,5,fixed-size,3120,4020,"ersion, make a model release and keep an updat...",900


In [27]:
import pymupdf

pdf_path = "clgdataset.pdf"
doc = pymupdf.open(pdf_path)
HEADER_HEIGHT = 30

for page in doc:
    page.set_cropbox(
        pymupdf.Rect(
            page.rect.x0,
            page.rect.y0 + HEADER_HEIGHT,
            page.rect.x1,
            page.rect.y1
        )
    )

cleaned_pages = []

for page_num, page in enumerate(doc, start=1):
    text = page.get_text("text")
    cleaned_pages.append({
        "page": page_num,
        "text": text
    })

print(cleaned_pages[22]["text"])

Search based Solutions
Goal based Agents
The simplest agents were the reflex agents
base their actions on a direct mapping from states to actions
cannot operate well in environments
this mapping would be too large to store
would take too long to learn
no idea about objective or learning
We will focus on goal based agents
consider future actions
desirability of their outcomes
They consider
State centric representation
What is a State?
All information about the environment
This may become prohibitively huge
All information necessary to make a decision for the task at hand.
23 / 27



In [28]:
import pymupdf
import pandas as pd

HEADER_HEIGHT = 30
CHUNK_SIZE = 900
OVERLAP = 120

cropped_pages = []

with pymupdf.open("clgdataset.pdf") as doc:
    for page_num, page in enumerate(doc, start=1):
        page_rect = page.rect

        clip = pymupdf.Rect(
            page_rect.x0,
            page_rect.y0 + HEADER_HEIGHT,
            page_rect.x1,
            page_rect.y1
        )

        text = page.get_text("text", clip=clip)
        text = " ".join(text.replace("\n", " ").split())

        cropped_pages.append({
            "page": page_num,
            "text": text
        })

full_text = "\n\n".join(
    page["text"] for page in cropped_pages if page["text"].strip()
)

cropped_chunks = []
start = 0

while start < len(full_text):
    end = min(start + CHUNK_SIZE, len(full_text))
    chunk_text = full_text[start:end]

    cropped_chunks.append({
        "document": "clgdataset.pdf",
        "version": "cropped_header_30pt",
        "chunk_id": len(cropped_chunks) + 1,
        "method": "fixed-size",
        "start_char": start,
        "end_char": end,
        "text": chunk_text,
        "char_count": len(chunk_text)
    })

    if end == len(full_text):
        break

    start += CHUNK_SIZE - OVERLAP

cropped_fixed_size_df = pd.DataFrame(cropped_chunks)

for _, chunk in cropped_fixed_size_df.iterrows():
    print(f"\n--- CROPPED CHUNK {chunk['chunk_id']} ---")
    print(f"Characters: {chunk['char_count']}")
    print("-" * 80)
    print(chunk["text"])
    print("-" * 80)

cropped_fixed_size_df.to_csv(
    "clgdataset_fixed_size_cropped.csv",
    index=False
)

print(f"\nTotal cropped chunks: {len(cropped_fixed_size_df)}")
print("Saved: clgdataset_fixed_size_cropped.csv")


--- CROPPED CHUNK 1 ---
Characters: 900
--------------------------------------------------------------------------------
Agents and Environments Roshni Chakraborty Dept. Of Information Technology ABV-IIITM Gwalior September 1, 2026 1 / 27

Outline I 1 Definition and philosophy Quantify Intelligence Thinking Humanly 2 Rational Agents Ideal Rational Agents Vaccum Cleaner Agent 3 Nature of Environment Goal based Agents 2 / 27

Artificial Intelligence Definition and philosophy How to quantify intelligence ? Are humans intelligent? replicating human behavior is an early hallmark of intelligence Are humans always intelligent? Can non-human behavior be intelligent? Definition The study of how to make computers do things at which, at the moment, people are better The branch of computer science that is concerned with the automation of intelligent behavior Objectives Systems that think like humans/rationally Systems that act like humans /rationally 3 / 27

Artificial Intelligence Thinking Human

In [31]:
import pandas as pd
import pymupdf
from IPython.display import display

MIN_PARAGRAPH_SIZE = 250
MAX_CHUNK_SIZE = 900
OVERLAP_SIZE = 120

PDF_FILES = ["clgdataset.pdf", "kaggle.pdf"]


def extract_paragraphs(pdf_path):
    paragraphs = []

    with pymupdf.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")

            page_paragraphs = [
                p.strip()
                for p in text.split("\n\n")
                if p.strip()
            ]

            for paragraph in page_paragraphs:
                paragraph = " ".join(paragraph.split())

                if paragraph:
                    paragraphs.append({
                        "text": paragraph,
                        "page": page_num
                    })

    return paragraphs


def merge_small_paragraphs(paragraphs):
    merged = []

    for paragraph in paragraphs:
        if merged and len(merged[-1]["text"]) < MIN_PARAGRAPH_SIZE:
            merged[-1]["text"] += "\n\n" + paragraph["text"]
            merged[-1]["end_page"] = paragraph["page"]
        else:
            merged.append({
                "text": paragraph["text"],
                "start_page": paragraph["page"],
                "end_page": paragraph["page"]
            })

    if len(merged) > 1 and len(merged[-1]["text"]) < MIN_PARAGRAPH_SIZE:
        merged[-2]["text"] += "\n\n" + merged[-1]["text"]
        merged[-2]["end_page"] = merged[-1]["end_page"]
        merged.pop()

    return merged


def split_long_paragraph(text):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + MAX_CHUNK_SIZE, len(text))

        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary

        chunks.append(text[start:end].strip())

        if end >= len(text):
            break

        start = max(start + 1, end - OVERLAP_SIZE)

    return chunks


def paragraph_chunking(pdf_path):
    paragraphs = extract_paragraphs(pdf_path)
    merged_paragraphs = merge_small_paragraphs(paragraphs)

    chunks = []

    for paragraph in merged_paragraphs:
        text = paragraph["text"]

        if len(text) > MAX_CHUNK_SIZE:
            pieces = split_long_paragraph(text)
        else:
            pieces = [text]

        for piece in pieces:
            chunks.append({
                "text": piece,
                "char_count": len(piece),
                "start_page": paragraph["start_page"],
                "end_page": paragraph["end_page"]
            })

    return chunks


paragraph_results = []

for filename in PDF_FILES:
    chunks = paragraph_chunking(filename)

    print("\n" + "=" * 100)
    print(f"DOCUMENT: {filename}")
    print(f"TOTAL CHUNKS: {len(chunks)}")
    print("=" * 100)

    for chunk_id, chunk in enumerate(chunks, start=1):
        record = {
            "document": filename,
            "chunk_id": chunk_id,
            "method": "paragraph-based",
            "start_page": chunk["start_page"],
            "end_page": chunk["end_page"],
            "text": chunk["text"],
            "char_count": chunk["char_count"]
        }

        paragraph_results.append(record)

        print(f"\n--- CHUNK {chunk_id} ---")
        print(f"Characters: {chunk['char_count']}")
        print(
            f"Page(s): {chunk['start_page']}–{chunk['end_page']}"
        )
        print("-" * 80)
        print(chunk["text"])
        print("-" * 80)

paragraph_chunks_df = pd.DataFrame(paragraph_results)

paragraph_chunks_df.to_csv(
    "paragraph_chunks.csv",
    index=False
)

for filename in PDF_FILES:
    output_name = filename.replace(
        ".pdf", "_paragraph_chunks.csv"
    )

    paragraph_chunks_df[
        paragraph_chunks_df["document"] == filename
    ].to_csv(output_name, index=False)

print("\n" + "=" * 100)
print("CSV SAVED: paragraph_chunks.csv")
print(
    f"TOTAL CHUNKS ACROSS BOTH DOCUMENTS: "
    f"{len(paragraph_chunks_df)}"
)
print("=" * 100)

print("\nFIRST 5 ROWS OF THE CSV")
display(paragraph_chunks_df.head(5))



DOCUMENT: clgdataset.pdf
TOTAL CHUNKS: 23

--- CHUNK 1 ---
Characters: 433
Page(s): 1–2
--------------------------------------------------------------------------------
Definition and philosophy Rational Agents Nature of Environment Agents and Environments Roshni Chakraborty Dept. Of Information Technology ABV-IIITM Gwalior September 1, 2026 1 / 27

Definition and philosophy Rational Agents Nature of Environment Outline I 1 Definition and philosophy Quantify Intelligence Thinking Humanly 2 Rational Agents Ideal Rational Agents Vaccum Cleaner Agent 3 Nature of Environment Goal based Agents 2 / 27
--------------------------------------------------------------------------------

--- CHUNK 2 ---
Characters: 634
Page(s): 3–3
--------------------------------------------------------------------------------
Definition and philosophy Rational Agents Nature of Environment Quantify Intelligence Thinking Humanly Artificial Intelligence Definition and philosophy How to quantify intelligence ? Are 

,document,chunk_id,method,start_page,end_page,text,char_count
0,clgdataset.pdf,1,paragraph-based,1,2,Definition and philosophy Rational Agents Natu...,433
1,clgdataset.pdf,2,paragraph-based,3,3,Definition and philosophy Rational Agents Natu...,634
2,clgdataset.pdf,3,paragraph-based,4,4,Definition and philosophy Rational Agents Natu...,462
3,clgdataset.pdf,4,paragraph-based,5,5,Definition and philosophy Rational Agents Natu...,622
4,clgdataset.pdf,5,paragraph-based,6,6,Definition and philosophy Rational Agents Natu...,574


In [45]:
paragraph_chunks_df = pd.DataFrame(paragraph_results)

kaggle_paragraph_chunks_df = paragraph_chunks_df[
    paragraph_chunks_df["document"] == "kaggle.pdf"
].copy()

kaggle_paragraph_chunks_df.to_csv(
    "kaggle_paragraph_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("CSV SAVED: kaggle_paragraph_chunks.csv")
print(f"TOTAL KAGGLE CHUNKS: {len(kaggle_paragraph_chunks_df)}")
display(kaggle_paragraph_chunks_df.head(5))

CSV SAVED: kaggle_paragraph_chunks.csv
TOTAL KAGGLE CHUNKS: 11


,document,chunk_id,method,start_page,end_page,text,char_count
23,kaggle.pdf,1,paragraph-based,1,2,Peer Review Framework for Predictive Analytics...,898
24,kaggle.pdf,2,paragraph-based,1,2,"se before a typhoon hit, based on the predicte...",899
25,kaggle.pdf,3,paragraph-based,1,2,odel will trigger or inform as part of the 202...,424
26,kaggle.pdf,4,paragraph-based,3,3,Intended Use ● The target of the model is P (#...,899
27,kaggle.pdf,5,paragraph-based,3,3,orities to estimate damage. ● Specify how much...,892


In [32]:
import pandas as pd
import pymupdf
from IPython.display import display

MIN_PARAGRAPH_SIZE = 250
MAX_CHUNK_SIZE = 900
OVERLAP_SIZE = 120
CROP_TOP = 30

PDF_FILE = "clgdataset.pdf"


def extract_paragraphs_cropped(pdf_path):
    paragraphs = []

    with pymupdf.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):

            clip = pymupdf.Rect(
                0,
                CROP_TOP,
                page.rect.width,
                page.rect.height
            )

            text = page.get_text("text", clip=clip)

            page_paragraphs = [
                p.strip()
                for p in text.split("\n\n")
                if p.strip()
            ]

            for paragraph in page_paragraphs:
                paragraph = " ".join(paragraph.split())

                if paragraph:
                    paragraphs.append({
                        "text": paragraph,
                        "page": page_num
                    })

    return paragraphs


def merge_small_paragraphs(paragraphs):
    merged = []

    for paragraph in paragraphs:
        if merged and len(merged[-1]["text"]) < MIN_PARAGRAPH_SIZE:
            merged[-1]["text"] += "\n\n" + paragraph["text"]
            merged[-1]["end_page"] = paragraph["page"]
        else:
            merged.append({
                "text": paragraph["text"],
                "start_page": paragraph["page"],
                "end_page": paragraph["page"]
            })

    if len(merged) > 1 and len(merged[-1]["text"]) < MIN_PARAGRAPH_SIZE:
        merged[-2]["text"] += "\n\n" + merged[-1]["text"]
        merged[-2]["end_page"] = merged[-1]["end_page"]
        merged.pop()

    return merged


def split_long_paragraph(text):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + MAX_CHUNK_SIZE, len(text))

        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary

        chunks.append(text[start:end].strip())

        if end >= len(text):
            break

        start = max(start + 1, end - OVERLAP_SIZE)

    return chunks


def paragraph_chunking_cropped(pdf_path):
    paragraphs = extract_paragraphs_cropped(pdf_path)
    merged_paragraphs = merge_small_paragraphs(paragraphs)

    chunks = []

    for paragraph in merged_paragraphs:
        text = paragraph["text"]

        if len(text) > MAX_CHUNK_SIZE:
            pieces = split_long_paragraph(text)
        else:
            pieces = [text]

        for piece in pieces:
            chunks.append({
                "text": piece,
                "char_count": len(piece),
                "start_page": paragraph["start_page"],
                "end_page": paragraph["end_page"]
            })

    return chunks

chunks = paragraph_chunking_cropped(PDF_FILE)

cropped_paragraph_results = []

print("\n" + "=" * 100)
print(f"DOCUMENT: {PDF_FILE} (CROPPED)")
print(f"TOTAL CHUNKS: {len(chunks)}")
print("=" * 100)

for chunk_id, chunk in enumerate(chunks, start=1):
    record = {
        "document": PDF_FILE,
        "chunk_id": chunk_id,
        "method": "paragraph-based-cropped",
        "start_page": chunk["start_page"],
        "end_page": chunk["end_page"],
        "text": chunk["text"],
        "char_count": chunk["char_count"]
    }

    cropped_paragraph_results.append(record)

    print(f"\n--- CHUNK {chunk_id} ---")
    print(f"Characters: {chunk['char_count']}")
    print(
        f"Page(s): {chunk['start_page']}–{chunk['end_page']}"
    )
    print("-" * 80)
    print(chunk["text"])
    print("-" * 80)


cropped_paragraph_df = pd.DataFrame(cropped_paragraph_results)

cropped_paragraph_df.to_csv(
    "clgdataset_paragraph_chunks_cropped.csv",
    index=False
)

print("\n" + "=" * 100)
print("CSV SAVED: clgdataset_paragraph_chunks_cropped.csv")
print(f"TOTAL CROPPED CHUNKS: {len(cropped_paragraph_df)}")
print("=" * 100)

print("\nFIRST 5 ROWS OF THE CSV")
display(cropped_paragraph_df.head(5))



DOCUMENT: clgdataset.pdf (CROPPED)
TOTAL CHUNKS: 22

--- CHUNK 1 ---
Characters: 305
Page(s): 1–2
--------------------------------------------------------------------------------
Agents and Environments Roshni Chakraborty Dept. Of Information Technology ABV-IIITM Gwalior September 1, 2026 1 / 27

Outline I 1 Definition and philosophy Quantify Intelligence Thinking Humanly 2 Rational Agents Ideal Rational Agents Vaccum Cleaner Agent 3 Nature of Environment Goal based Agents 2 / 27
--------------------------------------------------------------------------------

--- CHUNK 2 ---
Characters: 531
Page(s): 3–3
--------------------------------------------------------------------------------
Artificial Intelligence Definition and philosophy How to quantify intelligence ? Are humans intelligent? replicating human behavior is an early hallmark of intelligence Are humans always intelligent? Can non-human behavior be intelligent? Definition The study of how to make computers do things at which, a

,document,chunk_id,method,start_page,end_page,text,char_count
0,clgdataset.pdf,1,paragraph-based-cropped,1,2,Agents and Environments Roshni Chakraborty Dep...,305
1,clgdataset.pdf,2,paragraph-based-cropped,3,3,Artificial Intelligence Definition and philoso...,531
2,clgdataset.pdf,3,paragraph-based-cropped,4,4,Artificial Intelligence Thinking Humanly Cogni...,359
3,clgdataset.pdf,4,paragraph-based-cropped,5,5,Artificial Intelligence Thinking Humanly Laws ...,519
4,clgdataset.pdf,5,paragraph-based-cropped,6,6,Artificial Intelligence Rational Agents An age...,467


In [41]:

import pymupdf
import re
import pandas as pd
from collections import Counter
from IPython.display import display

PDF_FILES = ["clgdataset.pdf", "kaggle.pdf"]
CROP_TOP = 30

def get_clip(page, filename):
    if filename == "clgdataset.pdf":
        return pymupdf.Rect(
            0, CROP_TOP, page.rect.width, page.rect.height
        )

    return page.rect

def reconstruct_line_text(spans):
    spans = sorted(spans, key=lambda span: span["bbox"][0])

    result = ""
    previous_span = None

    for span in spans:
        text = span.get("text", "")

        if not text:
            continue

        if previous_span is not None:
            previous_end = previous_span["bbox"][2]
            current_start = span["bbox"][0]
            gap = current_start - previous_end

            average_size = (
                previous_span.get("size", 10)
                + span.get("size", 10)
            ) / 2

            if (
                gap > max(1.0, average_size * 0.20)
                and not result.endswith(" ")
                and not text.startswith(" ")
            ):
                result += " "

        result += text
        previous_span = span

    return result.strip()

def get_rgb(color):
    return (
        (color >> 16) & 255,
        (color >> 8) & 255,
        color & 255
    )


def is_distinctive_color(color):
    """
    Detect saturated colors and white/light text.
    Ordinary black and dark-gray text is not distinctive.
    """
    r, g, b = get_rgb(color)

    brightness = (r + g + b) / 3
    chroma = max(r, g, b) - min(r, g, b)

    return chroma > 25 or brightness > 200

def is_bold(span):
    font_name = span.get("font", "").lower()
    flags = span.get("flags", 0)

    # PyMuPDF flag bit 4 indicates bold.
    return bool(flags & 16) or any(
        word in font_name
        for word in ["bold", "black", "heavy"]
    )


def is_italic(span):
    font_name = span.get("font", "").lower()
    flags = span.get("flags", 0)

    # PyMuPDF flag bit 1 indicates italic.
    return bool(flags & 2) or any(
        word in font_name
        for word in ["italic", "oblique"]
    )

def extract_lines(page, filename, page_number):
    clip = get_clip(page, filename)

    page_dict = page.get_text(
        "dict",
        clip=clip,
        sort=True
    )

    lines = []

    for block in page_dict.get("blocks", []):
        if "lines" not in block:
            continue

        for line in block["lines"]:
            spans = [
                span
                for span in line.get("spans", [])
                if span.get("text", "").strip()
            ]

            if not spans:
                continue

            text = reconstruct_line_text(spans)

            if not text:
                continue

            total_chars = sum(
                len(span.get("text", "").strip())
                for span in spans
            )

            bold_chars = sum(
                len(span.get("text", "").strip())
                for span in spans
                if is_bold(span)
            )

            italic_chars = sum(
                len(span.get("text", "").strip())
                for span in spans
                if is_italic(span)
            )

            bold_ratio = bold_chars / max(1, total_chars)
            italic_ratio = italic_chars / max(1, total_chars)

            lines.append({
                "page": page_number,
                "text": text,
                "x": line["bbox"][0],
                "y": line["bbox"][1],
                "x1": line["bbox"][2],
                "y1": line["bbox"][3],
                "font_size": max(
                    span.get("size", 0) for span in spans
                ),
                "bold_ratio": bold_ratio,
                "italic_ratio": italic_ratio,
                "distinctive_color": any(
                    is_distinctive_color(span.get("color", 0))
                    for span in spans
                ),
            })

    return lines


def estimate_body_font_size(lines):
    counts = Counter()

    for line in lines:
        size = round(line["font_size"], 1)
        counts[size] += max(1, len(line["text"]))

    if not counts:
        return 0

    return counts.most_common(1)[0][0]


def is_heading_candidate(line, body_size):
    text = line["text"].strip()

    if not text or len(text) > 140:
        return False

    if re.fullmatch(r"[\d\s.()\-]+", text):
        return False

    if re.match(r"^[•●▪◦]\s+", text):
        return False

    if re.search(r"[.!?;,:]\s*$", text):
        return False

    larger_font = (
        body_size > 0
        and line["font_size"] >= body_size * 1.10
    )

    bold = line["bold_ratio"] >= 0.80
    italic = line["italic_ratio"] >= 0.80
    colored = line["distinctive_color"]

    numbered_colored = (
        colored
        and re.match(
            r"^\s*\d+(?:\.\d+)*\.?(?:\s+|$)",
            text
        )
    )

    if numbered_colored:
        return True

    return (
        (larger_font and bold)
        or (larger_font and colored)
        or (larger_font and italic)
        or (bold and colored)
        or (bold and italic)
        or (colored and italic)
    )

all_heading_chunks = []
heading_chunks_by_file = {}

for filename in PDF_FILES:

    print("\n" + "=" * 100)
    print(f"DOCUMENT: {filename}")
    print("=" * 100)

    doc = pymupdf.open(filename)
    document_lines = []

    for page_index, page in enumerate(doc):
        page_number = page_index + 1

        page_lines = extract_lines(
            page,
            filename,
            page_number
        )

        body_size = estimate_body_font_size(page_lines)

        for line in page_lines:
            line["is_heading"] = is_heading_candidate(
                line,
                body_size
            )

        document_lines.extend(page_lines)

    doc.close()

    sections = []
    current_heading = None
    current_content = []
    current_pages = []

    def save_section():
        content_text = "\n".join(
            item["text"] for item in current_content
        ).strip()

        if current_heading is None and not content_text:
            return

        if current_heading is not None:
            heading_text = current_heading["text"]
            full_text = heading_text

            if content_text:
                full_text += "\n" + content_text

            heading_page = current_heading["page"]

        else:
            heading_text = "Content before first heading"
            full_text = content_text
            heading_page = (
                min(current_pages) if current_pages else None
            )

        page_numbers = sorted(set(current_pages))

        section = {
            "document": filename,
            "heading": heading_text,
            "heading_page": heading_page,
            "pages": ", ".join(map(str, page_numbers)),
            "char_count": len(full_text),
            "text": full_text,
        }

        # Merge consecutive identical headings.
        if (
            sections
            and current_heading is not None
            and sections[-1]["heading"].casefold()
                == heading_text.casefold()
        ):
            previous = sections[-1]

            if content_text:
                previous["text"] += "\n" + content_text

            previous["char_count"] = len(previous["text"])

            old_pages = {
                int(p.strip())
                for p in previous["pages"].split(",")
                if p.strip().isdigit()
            }

            old_pages.update(page_numbers)

            previous["pages"] = ", ".join(
                map(str, sorted(old_pages))
            )

        else:
            sections.append(section)

    for line in document_lines:

        if line["is_heading"]:
            save_section()

            current_heading = line
            current_content = []
            current_pages = [line["page"]]

        else:
            current_content.append(line)
            current_pages.append(line["page"])

    save_section()

    heading_chunks_by_file[filename] = sections
    all_heading_chunks.extend(sections)

    print(f"TOTAL CHUNKS: {len(sections)}")

    for chunk_id, chunk in enumerate(sections, start=1):
        print("\n" + "-" * 100)
        print(f"CHUNK: {chunk_id}")
        print(f"HEADING: {chunk['heading']}")
        print(f"PAGES: {chunk['pages']}")
        print(f"CHARACTERS: {chunk['char_count']}")
        print("-" * 100)
        print(chunk["text"])
        print("-" * 100)

heading_chunks_df = pd.DataFrame(all_heading_chunks)

heading_chunks_df.to_csv(
    "heading_based_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: heading_based_chunks.csv")

for filename, chunks in heading_chunks_by_file.items():

    output_name = filename.replace(
        ".pdf",
        "_heading_chunks.csv"
    )

    pd.DataFrame(chunks).to_csv(
        output_name,
        index=False,
        encoding="utf-8"
    )

    print(f"Saved: {output_name}")


display(heading_chunks_df.head(10))



DOCUMENT: clgdataset.pdf
TOTAL CHUNKS: 36

----------------------------------------------------------------------------------------------------
CHUNK: 1
HEADING: Agents and Environments
PAGES: 1
CHARACTERS: 117
----------------------------------------------------------------------------------------------------
Agents and Environments
Roshni Chakraborty
Dept. Of Information Technology
ABV-IIITM Gwalior
September 1, 2026
1 / 27
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
CHUNK: 2
HEADING: Outline I
PAGES: 2
CHARACTERS: 186
----------------------------------------------------------------------------------------------------
Outline I
1
Definition and philosophy
Quantify Intelligence
Thinking Humanly
2
Rational Agents
Ideal Rational Agents
Vaccum Cleaner Agent
3
Nature of Environment
Goal based Agents
2 / 27
-------------------------

,document,heading,heading_page,pages,char_count,text
0,clgdataset.pdf,Agents and Environments,1,1,117,Agents and Environments\nRoshni Chakraborty\nD...
1,clgdataset.pdf,Outline I,2,2,186,Outline I\n1\nDefinition and philosophy\nQuant...
2,clgdataset.pdf,Artificial Intelligence,3,3,23,Artificial Intelligence
3,clgdataset.pdf,Definition and philosophy,3,3,507,Definition and philosophy\nHow to quantify int...
4,clgdataset.pdf,Artificial Intelligence,4,"4, 5, 6",855,Artificial Intelligence\nThinking Humanly\nCog...
5,clgdataset.pdf,Rational Agents,6,"6, 7",482,Rational Agents\nAn agent should strive to do ...
6,clgdataset.pdf,Artificial Intelligence,8,8,23,Artificial Intelligence
7,clgdataset.pdf,Ideal Rational Agent,8,8,578,Ideal Rational Agent\nFor each possible percep...
8,clgdataset.pdf,Artificial Intelligence,9,"9, 10, 11",430,Artificial Intelligence\nIdeal Rational Agent\...
9,clgdataset.pdf,Ideal Rational Agent,11,11,606,Ideal Rational Agent\nA rational agent is one ...


In [46]:
clg_heading_df = pd.DataFrame(
    heading_chunks_by_file["clgdataset.pdf"]
)

clg_heading_df.to_csv(
    "clgdataset_heading_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("CSV SAVED: clgdataset_heading_chunks.csv")
print(f"TOTAL CLG CHUNKS: {len(clg_heading_df)}")

kaggle_heading_df = pd.DataFrame(
    heading_chunks_by_file["kaggle.pdf"]
)

kaggle_heading_df.to_csv(
    "kaggle_heading_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("=" * 100)
print("CSV SAVED: kaggle_heading_chunks.csv")
print(f"TOTAL KAGGLE CHUNKS: {len(kaggle_heading_df)}")

print("\nFIRST 5 ROWS — CLG DATASET")
display(clg_heading_df.head(5))

print("\nFIRST 5 ROWS — KAGGLE")
display(kaggle_heading_df.head(5))

CSV SAVED: clgdataset_heading_chunks.csv
TOTAL CLG CHUNKS: 36
CSV SAVED: kaggle_heading_chunks.csv
TOTAL KAGGLE CHUNKS: 21

FIRST 5 ROWS — CLG DATASET


,document,heading,heading_page,pages,char_count,text
0,clgdataset.pdf,Agents and Environments,1,1,117,Agents and Environments\nRoshni Chakraborty\nD...
1,clgdataset.pdf,Outline I,2,2,186,Outline I\n1\nDefinition and philosophy\nQuant...
2,clgdataset.pdf,Artificial Intelligence,3,3,23,Artificial Intelligence
3,clgdataset.pdf,Definition and philosophy,3,3,507,Definition and philosophy\nHow to quantify int...
4,clgdataset.pdf,Artificial Intelligence,4,"4, 5, 6",855,Artificial Intelligence\nThinking Humanly\nCog...



FIRST 5 ROWS — KAGGLE


,document,heading,heading_page,pages,char_count,text
0,kaggle.pdf,Peer Review Framework for Predictive Analytics,1,1,46,Peer Review Framework for Predictive Analytics
1,kaggle.pdf,in Humanitarian Response,1,1,38,in Humanitarian Response\nMODEL REPORT:
2,kaggle.pdf,Typhoon impact model,1,1,20,Typhoon impact model
3,kaggle.pdf,510 - An initiative of the Netherlands Red Cross,1,"1, 2",74,510 - An initiative of the Netherlands Red Cro...
4,kaggle.pdf,Typhoon impact model,2,2,20,Typhoon impact model


In [51]:
import os
import zipfile
from google.colab import files

csv_files = [
    "kaggle_fixed_size_chunks.csv",
    "clgdataset_fixed_size_cropped.csv",
    "kaggle_paragraph_chunks.csv",
    "clgdataset_paragraph_chunks_cropped.csv",
    "clgdataset_heading_chunks.csv",
    "kaggle_heading_chunks.csv"
]

# Check that all six CSV files exist
missing_files = [
    filename for filename in csv_files
    if not os.path.exists(filename)
]

if missing_files:
    print("The following CSV files are missing:")
    for filename in missing_files:
        print(f"- {filename}")
    print("\nPlease run the corresponding save cells first.")

else:
    zip_filename = "all_6_chunking_csvs.zip"

    with zipfile.ZipFile(
        zip_filename, "w", zipfile.ZIP_DEFLATED
    ) as zipf:
        for filename in csv_files:
            zipf.write(filename, arcname=filename)


    print("ALL 6 CSV FILES ARE READY FOR DOWNLOAD")
    print(" " * 70)

    for filename in csv_files:
        print(f"✓ {filename}")

    print(f"\nZIP FILE: {zip_filename}")
    print("Starting download...")

    files.download(zip_filename)

ALL 6 CSV FILES ARE READY FOR DOWNLOAD
                                                                      
✓ kaggle_fixed_size_chunks.csv
✓ clgdataset_fixed_size_cropped.csv
✓ kaggle_paragraph_chunks.csv
✓ clgdataset_paragraph_chunks_cropped.csv
✓ clgdataset_heading_chunks.csv
✓ kaggle_heading_chunks.csv

ZIP FILE: all_6_chunking_csvs.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
import os
import re
import pandas as pd
import numpy as np
import pymupdf
from IPython.display import display

FILES = {
    "Kaggle": {
        "Fixed-size": "kaggle_fixed_size_chunks.csv",
        "Paragraph-based": "kaggle_paragraph_chunks.csv",
        "Heading-based": "kaggle_heading_chunks.csv",
    },
    "CLG dataset (cropped)": {
        "Fixed-size": "clgdataset_fixed_size_cropped.csv",
        "Paragraph-based": "clgdataset_paragraph_chunks_cropped.csv",
        "Heading-based": "clgdataset_heading_chunks.csv",
    },
}

TINY_LIMIT = 100
OVERSIZED_LIMIT = 1200
OVERLAP_SEARCH_LIMIT = 2000

def get_source_text(document):
    pdf_path = document
    if not os.path.exists(pdf_path):
        return None

    pages = []

    with pymupdf.open(pdf_path) as pdf:
        for page in pdf:
            if document == "clgdataset.pdf":
                clip = pymupdf.Rect(
                    0, 30, page.rect.width, page.rect.height
                )
                text = page.get_text("text", clip=clip)
            else:
                text = page.get_text("text")

            text = " ".join(text.split())
            if text:
                pages.append(text)

    return " ".join(pages)


source_texts = {
    "Kaggle": get_source_text("kaggle.pdf"),
    "CLG dataset (cropped)": get_source_text("clgdataset.pdf"),
}

def longest_adjacent_overlap(a, b):
    """Longest exact suffix of a matching the prefix of b."""
    a = a[-OVERLAP_SEARCH_LIMIT:]
    b = b[:OVERLAP_SEARCH_LIMIT]

    for length in range(min(len(a), len(b)), 0, -1):
        if a[-length:] == b[:length]:
            return length
    return 0


def starts_mid_word(text):
    """
    Conservative warning heuristic. Detects a chunk beginning
    with a lowercase continuation-like fragment after punctuation
    or whitespace has been removed.
    """
    text = text.lstrip()
    return bool(re.match(r"^[a-z]", text))


def ends_mid_word(text):
    """
    Conservative warning heuristic: lowercase endings can be
    normal, so use this alongside examples, not as a semantic score.
    """
    text = text.rstrip()
    return bool(re.search(r"[a-z]$", text))


def ends_with_continuation_punctuation(text):
    return bool(re.search(r"[,;:\-–—(/]$", text.rstrip()))


def looks_like_heading(line):
    line = line.strip()

    if not line or len(line) > 120:
        return False
    if re.search(r"[.!?;,:]$", line):
        return False

    numbered = bool(
        re.match(r"^\d+(?:\.\d+)*\.?\s+\S+", line)
    )
    uppercase = (
        line.isupper()
        and any(c.isalpha() for c in line)
    )

    words = line.split()
    title_case = (
        1 <= len(words) <= 12
        and sum(w[:1].isupper() for w in words if w)
        >= max(1, len(words) * 0.6)
    )

    return numbered or uppercase or title_case


def heading_flags(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    flags = [looks_like_heading(line) for line in lines]
    return flags

def calculate_metrics(document_label, method, filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"Missing {filename}. Run the corresponding save cell first."
        )

    df = pd.read_csv(filename).fillna("")

    if "text" not in df.columns:
        raise ValueError(f"No 'text' column in {filename}")

    df["text"] = df["text"].astype(str)
    df = df[df["text"].str.strip() != ""].reset_index(drop=True)

    texts = df["text"].tolist()
    sizes = [len(text) for text in texts]
    n = len(texts)
    total_chunk_chars = sum(sizes)

    tiny_count = sum(size < TINY_LIMIT for size in sizes)
    oversized_count = sum(size > OVERSIZED_LIMIT for size in sizes)

    overlap_chars = sum(
        longest_adjacent_overlap(texts[i], texts[i + 1])
        for i in range(n - 1)
    )

    overlap_percent = (
        100 * overlap_chars / total_chunk_chars
        if total_chunk_chars else 0
    )

    source_text = source_texts.get(document_label)
    if source_text:
        source_chars = len(source_text)
        source_ratio = total_chunk_chars / source_chars
    else:
        source_chars = np.nan
        source_ratio = np.nan

    starts_frag = sum(starts_mid_word(t) for t in texts)
    ends_frag = sum(ends_mid_word(t) for t in texts)
    boundary_frag = sum(
        starts_mid_word(t) or ends_mid_word(t)
        for t in texts
    )
    continuation = sum(
        ends_with_continuation_punctuation(t) for t in texts
    )

    starts_heading = 0
    middle_heading = 0

    for text in texts:
        flags = heading_flags(text)
        if flags and flags[0]:
            starts_heading += 1
        if any(flags[1:]):
            middle_heading += 1

    if "heading" in df.columns:
        headings = df["heading"].astype(str).str.strip()
        valid_headings = headings[
            (headings != "")
            & (headings.str.lower() != "content before first heading")
        ]
        heading_coverage = 100 * len(valid_headings) / max(1, n)
    else:
        heading_coverage = np.nan

    return {
        "Document": document_label,
        "Method": method,
        "Chunks": n,
        "Mean chars": round(np.mean(sizes), 1) if sizes else 0,
        "Median chars": round(np.median(sizes), 1) if sizes else 0,
        "Min chars": min(sizes) if sizes else 0,
        "Max chars": max(sizes) if sizes else 0,
        "Std dev chars": round(np.std(sizes), 1) if sizes else 0,
        f"Tiny chunks (<{TINY_LIMIT})": tiny_count,
        "Tiny chunks (%)": round(100 * tiny_count / n, 1) if n else 0,
        f"Oversized chunks (>{OVERSIZED_LIMIT})": oversized_count,
        "Oversized chunks (%)": round(100 * oversized_count / n, 1)
            if n else 0,
        "Adjacent overlap chars (est.)": overlap_chars,
        "Overlap / chunk chars (%)": round(overlap_percent, 2),
        "Chunk chars / source chars": round(source_ratio, 3)
            if not pd.isna(source_ratio) else np.nan,
        "Starts mid-word (%)": round(100 * starts_frag / n, 1)
            if n else 0,
        "Ends mid-word (%)": round(100 * ends_frag / n, 1)
            if n else 0,
        "Boundary-fragment chunks (%)": round(100 * boundary_frag / n, 1)
            if n else 0,
        "Continuation punctuation (%)": round(100 * continuation / n, 1)
            if n else 0,
        "Starts with heading (%)": round(100 * starts_heading / n, 1)
            if n else 0,
        "Heading in middle (%)": round(100 * middle_heading / n, 1)
            if n else 0,
        "Heading coverage (%)": round(heading_coverage, 1)
            if not pd.isna(heading_coverage) else np.nan,
    }

all_metrics = []

for document_label, methods in FILES.items():
    print("\n" + "=" * 110)
    print(f"DOCUMENT: {document_label}")
    print("=" * 110)

    rows = []

    for method, filename in methods.items():
        result = calculate_metrics(document_label, method, filename)
        rows.append(result)
        all_metrics.append(result)

    comparison = pd.DataFrame(rows)

    columns_to_show = [
        "Method",
        "Chunks",
        "Mean chars",
        "Median chars",
        "Min chars",
        "Max chars",
        "Tiny chunks (%)",
        "Oversized chunks (%)",
        "Adjacent overlap chars (est.)",
        "Overlap / chunk chars (%)",
        "Chunk chars / source chars",
        "Boundary-fragment chunks (%)",
        "Starts with heading (%)",
        "Heading in middle (%)",
        "Heading coverage (%)",
    ]

    display(comparison[columns_to_show])

metrics_df = pd.DataFrame(all_metrics)

metrics_df.to_csv(
    "chunking_comparison_metrics.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: chunking_comparison_metrics.csv")
display(metrics_df)



DOCUMENT: Kaggle


,Method,Chunks,Mean chars,Median chars,Min chars,Max chars,Tiny chunks (%),Oversized chunks (%),Adjacent overlap chars (est.),Overlap / chunk chars (%),Chunk chars / source chars,Boundary-fragment chunks (%),Starts with heading (%),Heading in middle (%),Heading coverage (%)
0,Fixed-size,11,837.6,900.0,214,900,0.0,0.0,1200,13.02,1.150,100.0,9.1,90.9,NaN
1,Paragraph-based,11,804.3,892.0,424,899,0.0,0.0,839,9.48,1.104,90.9,0.0,0.0,NaN
2,Heading-based,21,380.5,324.0,18,893,33.3,0.0,2,0.03,0.998,28.6,90.5,0.0,100.0



DOCUMENT: CLG dataset (cropped)


,Method,Chunks,Mean chars,Median chars,Min chars,Max chars,Tiny chunks (%),Oversized chunks (%),Adjacent overlap chars (est.),Overlap / chunk chars (%),Chunk chars / source chars,Boundary-fragment chunks (%),Starts with heading (%),Heading in middle (%),Heading coverage (%)
0,Fixed-size,15,886.7,900.0,701,900,0.0,0.0,1680,12.63,1.147,93.3,0.0,26.7,NaN
1,Paragraph-based,22,526.3,543.0,305,804,0.0,0.0,0,0.00,0.999,0.0,0.0,0.0,NaN
2,Heading-based,36,317.9,379.0,18,855,36.1,0.0,0,0.00,0.987,36.1,97.2,63.9,100.0



Saved: chunking_comparison_metrics.csv


,Document,Method,Chunks,Mean chars,Median chars,Min chars,Max chars,Std dev chars,Tiny chunks (<100),Tiny chunks (%),...,Adjacent overlap chars (est.),Overlap / chunk chars (%),Chunk chars / source chars,Starts mid-word (%),Ends mid-word (%),Boundary-fragment chunks (%),Continuation punctuation (%),Starts with heading (%),Heading in middle (%),Heading coverage (%)
0,Kaggle,Fixed-size,11,837.6,900.0,214,900,197.2,0,0.0,...,1200,13.02,1.150,81.8,90.9,100.0,0.0,9.1,90.9,NaN
1,Kaggle,Paragraph-based,11,804.3,892.0,424,899,151.9,0,0.0,...,839,9.48,1.104,63.6,63.6,90.9,0.0,0.0,0.0,NaN
2,Kaggle,Heading-based,21,380.5,324.0,18,893,306.3,7,33.3,...,2,0.03,0.998,4.8,23.8,28.6,9.5,90.5,0.0,100.0
3,CLG dataset (cropped),Fixed-size,15,886.7,900.0,701,900,49.6,0,0.0,...,1680,12.63,1.147,86.7,73.3,93.3,6.7,0.0,26.7,NaN
4,CLG dataset (cropped),Paragraph-based,22,526.3,543.0,305,804,126.3,0,0.0,...,0,0.00,0.999,0.0,0.0,0.0,0.0,0.0,0.0,NaN
5,CLG dataset (cropped),Heading-based,36,317.9,379.0,18,855,258.7,13,36.1,...,0,0.00,0.987,0.0,36.1,36.1,0.0,97.2,63.9,100.0


In [53]:
from google.colab import files

files.download("chunking_comparison_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
import pandas as pd
from IPython.display import display, HTML

SELECTIONS = [
    {"document": "Kaggle", "method": "Paragraph",
     "file": "kaggle_paragraph_chunks.csv", "chunk": 11, "label": "Good"},
    {"document": "Kaggle", "method": "Heading",
     "file": "kaggle_heading_chunks.csv", "chunk": 2, "label": "Bad"},
    {"document": "CLG", "method": "Heading",
     "file": "clgdataset_heading_chunks.csv", "chunk": 22, "label": "Good"},
    {"document": "CLG", "method": "Fixed-size",
     "file": "clgdataset_fixed_size_cropped.csv", "chunk": 6, "label": "Good"},
    {"document": "Kaggle", "method": "Fixed-size",
     "file": "kaggle_fixed_size_chunks.csv", "chunk": 3, "label": "Bad"},
    {"document": "CLG", "method": "Heading",
     "file": "clgdataset_heading_chunks.csv", "chunk": 3, "label": "Bad"},
]

cache = {}
rows = []

for item in SELECTIONS:
    filename = item["file"]

    if filename not in cache:
        cache[filename] = pd.read_csv(filename).fillna("")

    df = cache[filename]
    text_col = next(
        (c for c in ["text", "chunk_text", "content", "chunk"]
         if c in df.columns),
        None
    )

    if text_col is None:
        raise ValueError(f"No text column found in {filename}: {list(df.columns)}")

    idx = item["chunk"]

    if idx < 0 or idx >= len(df):
        print(f"Chunk {item['chunk']} not found in {filename}")
        continue

    texts = df[text_col].astype(str).tolist()

    rows.append({
        "Assessment": item["label"],
        "Document": item["document"],
        "Chunking Method": item["method"],
        "Chunk Number": item["chunk"],
        "Previous Chunk": texts[idx - 1] if idx > 0 else "",
        "Selected Chunk": texts[idx],
        "Next Chunk": texts[idx + 1] if idx + 1 < len(texts) else "",
        "Chunk Length": len(texts[idx]),
    })

result = pd.DataFrame(rows)
result["_order"] = result["Assessment"].map({"Bad": 0, "Good": 1})
result = result.sort_values("_order", kind="stable").drop(columns="_order")
result = result.reset_index(drop=True)

result.to_csv("six_selected_chunk_examples.csv", index=False)

html = result.to_html(index=False, escape=True)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table border="1" style="border-collapse:collapse; width:100%; '
    'table-layout:fixed; font-size:13px;">'
)

html = html.replace(
    "<th>",
    '<th style="padding:8px; background:#eaeaea; '
    'border:1px solid #999; overflow-wrap:anywhere;">'
)

html = html.replace(
    "<td>",
    '<td style="padding:8px; border:1px solid #999; '
    'vertical-align:top; white-space:pre-wrap; '
    'overflow-wrap:anywhere; min-width:120px;">'
)

display(HTML(html))

print("Saved complete table to six_selected_chunk_examples.csv")

Assessment,Document,Chunking Method,Chunk Number,Previous Chunk,Selected Chunk,Next Chunk,Chunk Length
Bad,Kaggle,Heading,2,Peer Review Framework for Predictive Analytics,in Humanitarian Response\nMODEL REPORT:,Typhoon impact model,38
Bad,Kaggle,Fixed-size,3,"\nbefore a typhoon hit, based on the predicted number of houses damaged in the regions of Bicol\nand Eastern Visayas. To find out more about this application, please see the Anticipatory Action\nFramework for the Philippines.\nThis peer review was conducted between September 2021 and July 2022.\n2. Main Findings and Recommendations\nThe documentation regarding the model, its application and the review process can be found at\nthe following links:\n●\nThe Model Card describes version 1.0 of the model and was completed in September\n2021.\n●\nThe Model Evaluation Matrix was completed in October 2021 by a technical expert from\nLeiden University, and in February 2022 by the client.\n●\nThe Implementation Plan was completed in October 2021. It summarises the concrete\nactions that the model will trigger or inform as part of the 2021-2022 OCHA anticipatory\naction framework in the Philippines\n●\nThe Ethical Ma","l will trigger or inform as part of the 2021-2022 OCHA anticipatory\naction framework in the Philippines\n●\nThe Ethical Matrix aims to identify all stakeholders and potential issues regarding the\nintended use of the model. It was completed in November 2021 by an expert from the\nLondon School of Economics; and updated in July 2022.\nA summary of the main findings and recommendations is provided below.\n2.1 Technical Review\n\nIntended Use\n●\nThe target of the model is P (#damaged houses > threshold t). Several combinations of\nProbability P and threshold t are used as a trigger for early action. Make a clear\nstatement under which circumstance which combination is used. This will help with later\nevaluation of the model and transparency of its use.\nModel Development\n●\nAs mentioned in the ethical review, more information about the sources of the input data\nis desirable. The model should include info","in the ethical review, more information about the sources of the input data\nis desirable. The model should include information on how often the data sets are\nupdated, and how often the model is retrained to reflect these updates.\nModel Evaluation\n●\nThe model is compared against a damage curve that is currently used to estimate\ndamage. Adjust this benchmark to whichever model or algorithm is used by the\nPhilippine authorities to estimate damage.\n●\nSpecify how much variance in model performance is acceptable, i.e., with the model\nworking\non\naverage\nbetter\nthan\nthe\nbenchmark,\nhow\nmuch\nextreme\nover\n/\nunderestimation is acceptable for field use. More specifically, indicate how robust the\nmodel performance is over different typhoons.\nOperational Readiness\n●\nFor every model version, make a model release and keep an updated list of associated\ninput data. This will make reproducibility of earlie",900
Bad,CLG,Heading,3,Outline I\n1\nDefinition and philosophy\nQuantify Intelligence\nThinking Humanly\n2\nRational Agents\nIdeal Rational Agents\nVaccum Cleaner Agent\n3\nNature of Environment\nGoal based Agents\n2 / 27,Artificial Intelligence,"Definition and philosophy\nHow to quantify intelligence ?\nAre humans intelligent?\nreplicating human behavior is an early hallmark of intelligence\nAre humans always intelligent?\nCan non-human behavior be intelligent?\nDefinition\nThe study of how to make computers do things at which, at the\nmoment, people are better\nThe branch of computer science that is concerned with the\nautomation of intelligent behavior\nObjectives\nSystems that think like humans/rationally\nSystems that act like humans /rationally\n3 / 27",23
Good,Kaggle,Paragraph,11,"rtance / weightage of the variables in the model is unclear. ● Recommendation: The model requires clear documentation of the different datasets used, particularly the rati

Saved complete table to six_selected_chunk_examples.csv
